# Lab Assignment 3
Medical data acquisition, preprocessing and analysis

## Setup

In [ ]:
!pip -q install kagglehub wfdb

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import wfdb

from pathlib import Path
from PIL import Image, ImageFilter
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from IPython.display import display

## A. Tabular Data

In [ ]:
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
diabetes = pd.read_csv(url)

diabetes.head()

### Inspection

In [ ]:
print("Rows:", diabetes.shape[0])
print("Columns:", diabetes.shape[1])
display(diabetes.dtypes.to_frame("Data Type"))
display(diabetes.isna().sum().to_frame("Missing Values"))
diabetes.describe()

### Cleaning

In [ ]:
clinical_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

clean_diabetes = diabetes.copy()
clean_diabetes[clinical_zero_columns] = clean_diabetes[clinical_zero_columns].replace(0, np.nan)

display(clean_diabetes[clinical_zero_columns].isna().sum().to_frame("Invalid or Missing"))

for column in clinical_zero_columns:
    clean_diabetes[column] = clean_diabetes[column].fillna(clean_diabetes[column].median())

clean_diabetes.isna().sum()

### EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(
    data=clean_diabetes,
    x="Glucose",
    hue="Outcome",
    kde=True,
    ax=axes[0]
)

sns.boxplot(
    data=clean_diabetes,
    x="Outcome",
    y="BMI",
    ax=axes[1]
)

axes[0].set_title("Glucose Distribution")
axes[1].set_title("BMI by Outcome")

plt.tight_layout()
plt.show()

### Feature Engineering

In [ ]:
clean_diabetes["AgeGroup"] = pd.cut(
    clean_diabetes["Age"],
    bins=[0, 30, 45, 60, np.inf],
    labels=["18-30", "31-45", "46-60", "60+"]
)

clean_diabetes["BMI_Category"] = pd.cut(
    clean_diabetes["BMI"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

clean_diabetes["Glucose_BMI_Ratio"] = (
    clean_diabetes["Glucose"] / clean_diabetes["BMI"]
)

clean_diabetes[
    ["Age", "AgeGroup", "BMI", "BMI_Category", "Glucose_BMI_Ratio"]
].head()

### Hypothesis Test

In [ ]:
positive_glucose = clean_diabetes.loc[
    clean_diabetes["Outcome"] == 1,
    "Glucose"
]

negative_glucose = clean_diabetes.loc[
    clean_diabetes["Outcome"] == 0,
    "Glucose"
]

t_stat, t_p = ttest_ind(
    positive_glucose,
    negative_glucose,
    equal_var=False
)

print("T statistic:", round(t_stat, 4))
print("P value:", round(t_p, 6))

### Chi-square Test

In [ ]:
chi_table = pd.crosstab(
    clean_diabetes["BMI_Category"],
    clean_diabetes["Outcome"]
)

chi2, chi_p, dof, expected = chi2_contingency(chi_table)

display(chi_table)
print("Chi-square:", round(chi2, 4))
print("P value:", round(chi_p, 6))
print("Degrees of freedom:", dof)

### ANOVA

In [ ]:
anova_groups = [
    group["Glucose"].values
    for _, group in clean_diabetes.groupby("AgeGroup", observed=True)
]

f_stat, anova_p = f_oneway(*anova_groups)

print("F statistic:", round(f_stat, 4))
print("P value:", round(anova_p, 6))

### Feature Selection and Scaling

In [ ]:
feature_columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Glucose_BMI_Ratio"
]

X = clean_diabetes[feature_columns]
y = clean_diabetes["Outcome"]

scores = mutual_info_classif(X, y, random_state=42)

feature_scores = pd.Series(
    scores,
    index=feature_columns
).sort_values(ascending=False)

display(feature_scores.to_frame("Mutual Information"))

selected_features = feature_scores.head(5).index.tolist()

scaler = MinMaxScaler()
tabular_tensor = scaler.fit_transform(
    clean_diabetes[selected_features]
)

print("Selected features:", selected_features)
print("Scaled shape:", tabular_tensor.shape)

## B. Textual Data

In [ ]:
mts_path = kagglehub.dataset_download(
    "tboyle10/medicaltranscriptions"
)

mts_file = Path(mts_path) / "mtsamples.csv"
mtsamples = pd.read_csv(mts_file)

mtsamples.head()

### Inspection and EDA

In [ ]:
print("Rows:", mtsamples.shape[0])
print("Columns:", mtsamples.shape[1])
display(mtsamples.isna().sum().to_frame("Missing Values"))

mtsamples["medical_specialty"].value_counts().head(10).plot(
    kind="bar",
    figsize=(10, 4)
)

plt.title("Most Common Medical Specialties")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### PHI Removal and Cleaning

In [ ]:
text_df = mtsamples[
    ["medical_specialty", "transcription"]
].dropna().copy()

def remove_phi(text):
    text = str(text)
    text = re.sub(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        "[EMAIL]",
        text
    )
    text = re.sub(
        r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}\b",
        "[PHONE]",
        text
    )
    text = re.sub(
        r"\b(?:MRN|Medical Record Number)\s*[:#-]?\s*\w+",
        "[MRN]",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(
        r"\b(?:DOB|Date of Birth)\s*[:#-]?\s*[A-Za-z0-9/,-]+",
        "[DOB]",
        text,
        flags=re.IGNORECASE
    )
    return text

def clean_text(text):
    text = remove_phi(text).lower()
    text = re.sub(r"[^a-z0-9\[\]\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

text_df["clean_text"] = text_df["transcription"].apply(clean_text)
text_df["word_count"] = text_df["clean_text"].str.split().str.len()

text_df[["medical_specialty", "word_count", "clean_text"]].head()

### Text EDA

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(text_df["word_count"], bins=40)
plt.title("Transcription Length")
plt.xlabel("Words")
plt.show()

### Token Sequences

In [ ]:
text_sample = text_df["clean_text"].head(1500)

tokenizer = Tokenizer(
    num_words=5000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(text_sample)

sequences = tokenizer.texts_to_sequences(text_sample)
text_tensor = pad_sequences(
    sequences,
    maxlen=200,
    padding="post",
    truncating="post"
)

print("Token tensor shape:", text_tensor.shape)
print(text_tensor[0][:30])

### TF-IDF Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(text_sample)

mean_scores = np.asarray(
    tfidf_matrix.mean(axis=0)
).ravel()

terms = np.array(
    vectorizer.get_feature_names_out()
)

top_terms = pd.DataFrame({
    "Term": terms,
    "Score": mean_scores
}).sort_values(
    "Score",
    ascending=False
).head(20)

print("TF-IDF shape:", tfidf_matrix.shape)
top_terms

## C. Image Data

In [ ]:
xray_path = kagglehub.dataset_download(
    "paultimothymooney/chest-xray-pneumonia"
)

xray_root = Path(xray_path)

normal_dir = next(
    p for p in xray_root.rglob("NORMAL")
    if p.parent.name == "train"
)

pneumonia_dir = next(
    p for p in xray_root.rglob("PNEUMONIA")
    if p.parent.name == "train"
)

normal_images = sorted(normal_dir.glob("*.jpeg"))
pneumonia_images = sorted(pneumonia_dir.glob("*.jpeg"))

print("Training normal:", len(normal_images))
print("Training pneumonia:", len(pneumonia_images))

### Image Inspection

In [ ]:
sample_path = normal_images[0]
image = Image.open(sample_path)

print("File:", sample_path.name)
print("Size:", image.size)
print("Mode:", image.mode)
print("Metadata keys:", list(image.info.keys()))

plt.figure(figsize=(5, 5))
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()

### Metadata Anonymization

In [ ]:
anonymized = Image.fromarray(
    np.asarray(image.convert("L"))
)

anonymized.save(
    "xray_anonymized.jpg",
    format="JPEG"
)

clean_image = Image.open(
    "xray_anonymized.jpg"
)

print("Metadata keys after saving:", list(clean_image.info.keys()))

### Image EDA

In [ ]:
image_files = normal_images[:25] + pneumonia_images[:25]

sizes = [
    Image.open(path).size
    for path in image_files
]

size_df = pd.DataFrame(
    sizes,
    columns=["Width", "Height"]
)

display(size_df.describe())

plt.figure(figsize=(6, 4))
plt.scatter(
    size_df["Width"],
    size_df["Height"]
)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Dimensions")
plt.show()

### Noise Removal and Preprocessing

In [ ]:
denoised = clean_image.filter(
    ImageFilter.MedianFilter(size=3)
)

resized = denoised.resize(
    (224, 224)
)

image_array = np.asarray(
    resized,
    dtype=np.float32
) / 255.0

image_tensor = image_array[
    np.newaxis,
    ...,
    np.newaxis
]

print("Image tensor shape:", image_tensor.shape)
print("Minimum:", image_tensor.min())
print("Maximum:", image_tensor.max())

### Image Augmentation

In [ ]:
augmenter = ImageDataGenerator(
    rotation_range=7,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05,
    fill_mode="nearest"
)

augmented = next(
    augmenter.flow(
        image_tensor,
        batch_size=1,
        shuffle=False
    )
)[0, :, :, 0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(image_tensor[0, :, :, 0], cmap="gray")
axes[0].set_title("Processed")
axes[0].axis("off")

axes[1].imshow(augmented, cmap="gray")
axes[1].set_title("Augmented")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## D. Signal Data

In [ ]:
record = wfdb.rdrecord(
    "100",
    pn_dir="mitdb",
    sampto=3600
)

annotation = wfdb.rdann(
    "100",
    "atr",
    pn_dir="mitdb",
    sampto=3600
)

signal = record.p_signal[:, 0]
fs = record.fs
time = np.arange(len(signal)) / fs

print("Sampling frequency:", fs)
print("Signal shape:", signal.shape)
print("Annotations:", len(annotation.sample))

### ECG EDA

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(time, signal)
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("MIT-BIH Record 100")
plt.show()

### Noise Removal

In [ ]:
low = 0.5 / (fs / 2)
high = 40 / (fs / 2)

b, a = butter(
    4,
    [low, high],
    btype="band"
)

filtered_signal = filtfilt(
    b,
    a,
    signal
)

plt.figure(figsize=(12, 4))
plt.plot(time, signal, alpha=0.5, label="Raw")
plt.plot(time, filtered_signal, label="Filtered")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.legend()
plt.show()

### Signal Windows

In [ ]:
window_size = int(fs * 2)
step = int(fs)

signal_windows = np.array([
    filtered_signal[start:start + window_size]
    for start in range(
        0,
        len(filtered_signal) - window_size + 1,
        step
    )
])

window_mean = signal_windows.mean(
    axis=1,
    keepdims=True
)

window_std = signal_windows.std(
    axis=1,
    keepdims=True
)

signal_tensor = (
    signal_windows - window_mean
) / (window_std + 1e-8)

print("Signal tensor shape:", signal_tensor.shape)

### Signal Feature Extraction

In [ ]:
signal_features = pd.DataFrame({
    "Mean": signal_windows.mean(axis=1),
    "Std": signal_windows.std(axis=1),
    "RMS": np.sqrt(
        np.mean(signal_windows ** 2, axis=1)
    ),
    "PeakToPeak": np.ptp(
        signal_windows,
        axis=1
    )
})

signal_features.head()

## Final Model-ready Data

In [ ]:
print("Tabular:", tabular_tensor.shape)
print("Text:", text_tensor.shape)
print("Image:", image_tensor.shape)
print("Signal:", signal_tensor.shape)

## Conclusion
The four medical data types were acquired, cleaned and converted into model-ready forms. Clinical zero values were treated as missing, text was de-identified, image metadata was stripped, ECG noise was filtered, and modality-specific EDA and feature processing were performed.